## 1. Using Pretrained GloVe in PyTorch

In [2]:
import torch

# Path to downloaded GloVe file
glove_file = r"C:\Users\sufis\Downloads\glove.6B\glove.6B.300d.txt"

# Dictionary to store word -> vector
glove_embeddings = {}

with open(glove_file, "r", encoding="utf-8") as f:
    for line in f:
        values = line.split()

        word = values[0]
        vector = torch.tensor(
            [float(x) for x in values[1:]],
            dtype=torch.float32
        )

        glove_embeddings[word] = vector

print("Vocabulary size:", len(glove_embeddings))
print("Embedding dimension:", glove_embeddings["king"].shape)

c:\Users\sufis\myenvv\Lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Vocabulary size: 400000
Embedding dimension: torch.Size([300])


In [3]:
word = "king"

embedding = glove_embeddings[word]

print("Embedding for:", word)
print(embedding)

Embedding for: king
tensor([ 0.0034, -0.3461,  0.2814,  0.4838,  0.5947,  0.0130,  0.5398,  0.4823,
         0.2146, -1.0249, -0.3479, -0.7900, -0.1508,  0.6137,  0.0428,  0.1932,
         0.2546,  0.3253,  0.0570,  0.0633, -0.4944,  0.4734, -0.1676,  0.0456,
         0.3045, -0.3542, -0.3458, -0.2012,  0.2551,  0.0911,  0.0147, -0.0175,
        -0.2385,  0.4821, -0.9145, -0.3623,  0.3474,  0.0286, -0.0271, -0.0365,
        -0.0674, -0.2345, -0.1377,  0.3395,  0.1341, -0.1342,  0.4786, -0.1842,
         0.1071, -0.4583, -0.3609, -0.2260,  0.3288, -0.1364,  0.2313,  0.3427,
         0.4234,  0.4706,  0.4790,  0.0746,  0.3344,  0.1071, -0.1329,  0.5873,
         0.3862, -0.5224, -0.2203, -0.0723,  0.3227,  0.4423, -0.0374,  0.1832,
         0.0581,  0.2694,  0.3620,  0.1398,  0.0168, -0.3443,  0.4827,  0.2108,
         0.7562, -0.1309, -0.0257,  0.4339,  0.3389, -0.1644,  0.2682,  0.6877,
         0.3110, -0.2509,  0.0028, -0.3981, -0.4340,  0.0495, -0.4269, -0.0947,
         0.5692,  0.

In [8]:
import torch.nn.functional as F

word1 = "king"
word2 = "queen"

similarity = F.cosine_similarity(
    glove_embeddings[word1].unsqueeze(0),
    glove_embeddings[word2].unsqueeze(0)
)

print("Cosine similarity:", similarity.item())

Cosine similarity: 0.6336469650268555


## 2. Train Your Own GloVe Using PyTorch

In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
from collections import Counter

# --------------------------------------------------
# 1. Small training corpus
# --------------------------------------------------

sentences = [
    "i love natural language processing",
    "i love machine learning",
    "i love deep learning",
    "machine learning is interesting",
    "deep learning is interesting",
    "natural language processing is interesting",
    "i like machine learning",
    "i like deep learning"
]

# --------------------------------------------------
# 2. Create vocabulary
# --------------------------------------------------

words = []

for sentence in sentences:
    words.extend(sentence.lower().split())

vocab = sorted(set(words))

word_to_id = {word: i for i, word in enumerate(vocab)}
id_to_word = {i: word for word, i in word_to_id.items()}

vocab_size = len(vocab)

print("Vocabulary:")
print(word_to_id)
print("Vocabulary size:", vocab_size)

Vocabulary:
{'deep': 0, 'i': 1, 'interesting': 2, 'is': 3, 'language': 4, 'learning': 5, 'like': 6, 'love': 7, 'machine': 8, 'natural': 9, 'processing': 10}
Vocabulary size: 11


In [10]:
# --------------------------------------------------
# 3. Create co-occurrence matrix
# --------------------------------------------------

window_size = 2

cooccurrence = torch.zeros(
    vocab_size,
    vocab_size,
    dtype=torch.float32
)

for sentence in sentences:

    tokens = sentence.lower().split()

    for i, word in enumerate(tokens):

        word_id = word_to_id[word]

        start = max(0, i - window_size)
        end = min(len(tokens), i + window_size + 1)

        for j in range(start, end):

            if i == j:
                continue

            context_word = tokens[j]
            context_id = word_to_id[context_word]

            # Distance weighting
            distance = abs(i - j)

            cooccurrence[word_id, context_id] += 1.0 / distance

print(cooccurrence)

tensor([[0.0000, 1.0000, 0.0000, 0.5000, 0.0000, 3.0000, 1.0000, 1.0000, 0.0000,
         0.0000, 0.0000],
        [1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 2.0000, 3.0000, 1.0000,
         0.5000, 0.0000],
        [0.0000, 0.0000, 0.0000, 3.0000, 0.0000, 1.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.5000],
        [0.5000, 0.0000, 3.0000, 0.0000, 0.5000, 2.0000, 0.0000, 0.0000, 0.5000,
         0.0000, 1.0000],
        [0.0000, 0.0000, 0.0000, 0.5000, 0.0000, 0.0000, 0.0000, 0.5000, 0.0000,
         2.0000, 2.0000],
        [3.0000, 0.0000, 1.0000, 2.0000, 0.0000, 0.0000, 1.0000, 1.0000, 3.0000,
         0.0000, 0.0000],
        [1.0000, 2.0000, 0.0000, 0.0000, 0.0000, 1.0000, 0.0000, 0.0000, 1.0000,
         0.0000, 0.0000],
        [1.0000, 3.0000, 0.0000, 0.0000, 0.5000, 1.0000, 0.0000, 0.0000, 1.0000,
         1.0000, 0.0000],
        [0.0000, 1.0000, 0.0000, 0.5000, 0.0000, 3.0000, 1.0000, 1.0000, 0.0000,
         0.0000, 0.0000],
        [0.0000, 0.5000, 0.0000, 0.00

In [11]:
# --------------------------------------------------
# 4. GloVe model
# --------------------------------------------------

embedding_dim = 50


class GloVeModel(nn.Module):

    def __init__(self, vocab_size, embedding_dim):

        super().__init__()

        # Word embeddings
        self.word_embeddings = nn.Embedding(
            vocab_size,
            embedding_dim
        )

        # Context embeddings
        self.context_embeddings = nn.Embedding(
            vocab_size,
            embedding_dim
        )

        # Word bias
        self.word_bias = nn.Embedding(
            vocab_size,
            1
        )

        # Context bias
        self.context_bias = nn.Embedding(
            vocab_size,
            1
        )

    def forward(self, word_ids, context_ids):

        word_vectors = self.word_embeddings(word_ids)
        context_vectors = self.context_embeddings(context_ids)

        word_bias = self.word_bias(word_ids).squeeze()
        context_bias = self.context_bias(context_ids).squeeze()

        # Dot product
        score = (
            (word_vectors * context_vectors).sum(dim=1)
            + word_bias
            + context_bias
        )

        return score

In [12]:
# --------------------------------------------------
# 5. Create training pairs
# --------------------------------------------------

word_ids = []
context_ids = []
counts = []

for i in range(vocab_size):

    for j in range(vocab_size):

        if cooccurrence[i, j] > 0:

            word_ids.append(i)
            context_ids.append(j)
            counts.append(cooccurrence[i, j].item())

word_ids = torch.tensor(word_ids, dtype=torch.long)
context_ids = torch.tensor(context_ids, dtype=torch.long)
counts = torch.tensor(counts, dtype=torch.float32)

print("Number of training pairs:", len(counts))

Number of training pairs: 52


In [13]:
def glove_loss(prediction, counts):

    log_counts = torch.log(counts)

    # Weighting function
    x_max = 100
    alpha = 0.75

    weights = torch.where(
        counts < x_max,
        (counts / x_max) ** alpha,
        torch.ones_like(counts)
    )

    loss = weights * (prediction - log_counts) ** 2

    return loss.mean()

In [14]:
# --------------------------------------------------
# 7. Train
# --------------------------------------------------

model = GloVeModel(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim
)

optimizer = optim.Adam(
    model.parameters(),
    lr=0.05
)

epochs = 1000

for epoch in range(epochs):

    optimizer.zero_grad()

    predictions = model(
        word_ids,
        context_ids
    )

    loss = glove_loss(
        predictions,
        counts
    )

    loss.backward()

    optimizer.step()

    if (epoch + 1) % 100 == 0:

        print(
            f"Epoch {epoch+1}/{epochs}, "
            f"Loss: {loss.item():.4f}"
        )

Epoch 100/1000, Loss: 0.0000
Epoch 200/1000, Loss: 0.0000
Epoch 300/1000, Loss: 0.0000
Epoch 400/1000, Loss: 0.0000
Epoch 500/1000, Loss: 0.0000
Epoch 600/1000, Loss: 0.0000
Epoch 700/1000, Loss: 0.0000
Epoch 800/1000, Loss: 0.0000
Epoch 900/1000, Loss: 0.0000
Epoch 1000/1000, Loss: 0.0000


In [15]:
# --------------------------------------------------
# 8. Extract embeddings
# --------------------------------------------------

word_vectors = model.word_embeddings.weight.data
context_vectors = model.context_embeddings.weight.data

# Final GloVe representation
final_embeddings = word_vectors + context_vectors

In [16]:
word = "learning"

word_id = word_to_id[word]

embedding = final_embeddings[word_id]

print("Word:", word)
print("Embedding shape:", embedding.shape)
print("Embedding:")
print(embedding)

Word: learning
Embedding shape: torch.Size([50])
Embedding:
tensor([ 1.4210,  0.3634,  0.0697,  1.0545,  0.2868, -0.2190, -1.0535,  2.6234,
        -1.6923, -0.4712,  0.6294, -0.7513,  0.5426, -1.0462, -0.9990,  1.5914,
         0.9270,  1.3704,  0.1871,  0.3286, -0.1281, -0.1293,  0.3837, -0.1389,
        -0.8529, -0.5186,  0.6916,  0.5361,  0.7131, -1.7978,  2.7413, -0.8796,
        -0.1888, -0.2638,  0.0564, -0.2312,  0.8050,  1.3718, -0.1929, -1.6376,
        -1.0601,  0.3456,  0.8287,  0.3287, -0.9260, -0.8808,  1.9066, -0.8088,
         0.8608,  0.8465])
